## APIs in Python

#### Objectives

- Via the Python `wget` module we can obtain data from the web

- Good for updates and for proving provenance

- it requires to know the exact position of the data file

- all-or-nothing behaviour:

  - download everything from scratch again

  - download the whole file, select the needed data later

#### API: the idea

ask the web site/data server to serve an ad hoc file containing strictly the data we need

data is assigned directly to a file (CSV, JSON) so we can read it later

If connection breaks, or our request is too selective, the returned file could be empty

#### Example

We want to download 10 current articles on migration from `gdeltproject.org`.

The site only returns articles from the last three months, it's a rolling window.

Let us begin by asking the question directly on the browser, to check that the site is reachable and the query is not out-of-range (no article returned)

Try this on your browser:

[https://api.gdeltproject.org/api/v2/doc/doc?query=migration&mode=ArtList&maxrecords=10&format=json](https://api.gdeltproject.org/api/v2/doc/doc?query=migration&mode=ArtList&maxrecords=10&format=json)

Alternatively, try the `http` version:

[http://api.gdeltproject.org/api/v2/doc/doc?query=migration&mode=ArtList&maxrecords=10&format=json](http://api.gdeltproject.org/api/v2/doc/doc?query=migration&mode=ArtList&maxrecords=10&format=json)


#### Set up API sessions

In [ ]:
import os
import urllib.request
import wget


GDELT may refuse the default Python User-Agent, so let's dress up our connection as if it were a normal browser

In [ ]:
opener = urllib.request.build_opener()

opener.addheaders = [("User-Agent", "Mozilla/5.0")]

urllib.request.install_opener(opener)


In [ ]:
filename = "migration.json"

# wget would otherwise save a second copy as "migration (1).json"
if os.path.exists(filename):
    os.remove(filename)
    

In [ ]:
wget.download(url, out=filename)
# here `out=` fixes the name of the local file

print()  # insert a newline 

print(filename, os.path.getsize(filename), "bytes")
# here `os.path.getsize` verifies that the file is not empty


## The GDELT Project

#### *Global Database of Events, Language and Tone*

- a UN-supported site that monitors print, broadcast and web news from around the world, in over 100 languages
- open data, free and updated continuously
- The DOC 2.0 API searches a rolling window of three months of news coverage; no registration 
- no API key is required


(from VS Code, open the parent folder `78-APIs` to load the figures)

![](../imgs/gdelt-landing.png)

#### Background: API queries

See the Wikipedia definition of [Application Programming Interface (API)](https://en.wikipedia.org/wiki/API)

a defined menu of requests that our *client* may make to a *server,* together with the form of the responses.

The client never accesses the database directly; it uses only the operations that the API exposes.

#### Web APIs and HTTP

Web APIs use the **Hypertext Transfer Protocol (HTTP)**: the client sends a request to a URL, and the server returns a response with a status code and a body.

See again Wikipedia for [HTTP](https://en.wikipedia.org/wiki/HTTP) and [HTTP status codes](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes).

#### Anatomy of a request URL

```
https://api.gdeltproject.org/api/v2/doc/doc?query=migration&mode=ArtList&format=json
```

Parameters are written as `name=value` pairs separated by `&`.

Queries can be tried out on a browser to check for *liveness*

[URL (Wiki)](https://en.wikipedia.org/wiki/URL)

[Query string (Wiki)](https://en.wikipedia.org/wiki/Query_string)

## Request parameters

| Parameter | Meaning | Example |
|---|---|---|
| `query` | search terms/filters | `"migration" sourcelang:english` |
| `mode` | type of output | `ArtList`, `TimelineVol` |
| `maxrecords` | no. of articles (max 250) | `50` |
| `timespan` | look-back window | `1d`, `1week`, `3months` |
| `format` | output format | `json` |

#### What could possibly go wrong?

well, a lot.

![Failure paths](../imgs/06-failure-paths.svg)

#### JSON reminder

The JSON format for data files maps directly onto Python types.

| JSON | Python |
|---|---|
| object `{ }` | `dict` |
| array `[ ]` | `list` |
| string | `str` |
| number | `int` or `float` |
| `true`, `false`, `null` | `True`, `False`, `None` |

[JSON (Wiki)](https://en.wikipedia.org/wiki/JSON)

#### Reading JSON in Python

In [ ]:
import json

with open("migration.json") as f:
    data = json.load(f)

In [ ]:
print(type(data))              # <class 'dict'>
print(list(data.keys()))       # ['articles']
print(len(data["articles"]))   # number of articles returned
print(data["articles"][0])     # the first article, a dict

JSON is a Python list of dictionaries on wheels...

Example:

```json
{
  "articles": [
    {
      "url": "https://spectator.com/article/the-return-of-migration-centrism/",
      "url_mobile": "",
      "title": "Small boat crossings: what the quarterly figures show",
      "seendate": "20251117T093000Z",
      "socialimage": "",
      "domain": "spectator.com",
      "language": "English",
      "sourcecountry": "United Kingdom"
    },
    {
      "url": "https://spectator.com/article/frances-migration-hypocrisy/",
      "url_mobile": "",
      "title": "France’s migration hypocrisy",
      "seendate": "20260410T093000Z",
      "socialimage": "",
      "domain": "spectator.com",
      "language": "English",
      "sourcecountry": "United Kingdom"
    }
  ]
}
```

#### GDELT example: querying by country

The `query` parameter accepts search operators. 

`sourcecountry:UK` keeps only articles *published in* that country (to find articles *about* a country, put its name in the query instead, e.g. `migration Italy`).

GDELT uses FIPS 2-letter country codes, which differ from the usual ISO codes in places.  E.g., Germany is `GM`, not `DE`. 

A wrong code may return no articles, without any error.

Here we loop over a list of countries and save one JSON file for each.


In [ ]:
import time
import urllib.parse

BASE = "http://api.gdeltproject.org/api/v2/doc/doc"

countries = ["UK", "FR"]

`urlencode` writes the parameters as `name=value` pairs and handles spaces and special characters.


GDELT requires a delay of 5 or more seconds between successive queries, so we insert `time.sleep(6)` between queries.

In [ ]:
for country in countries:
    params = {
        "query": f"migration sourcecountry:{country}",
        "mode": "ArtList",
        "maxrecords": 10,
        "format": "json",
    }

    url = BASE + "?" + urllib.parse.urlencode(params)

    filename = f"migration-{country}.json"

    if os.path.exists(filename):
        os.remove(filename)

    wget.download(url, out=filename)
    
    print()
    print(country, os.path.getsize(filename), "bytes")

    time.sleep(6)   # GDELT asks for one request every 5 seconds

Now, read the files back and look at the results. 

Notice that `data.get("articles", [])` means that if there are no articles then it returs `[]`, thus avoiding a `KeyError`. 


In [ ]:
import json

for country in countries:
    with open(f"migration-{country}.json", encoding="utf-8") as f:
        data = json.load(f)

    articles = data.get("articles", [])

    print()
    print(country, len(articles), "articles")
    
    for article in articles:
        print("  ", article["seendate"][:8], article["domain"], "-", article["title"])